# Reachability Analysis - Ego Car NN Controller

## 1. Initial states and mapping to NN inputs

In [4]:
from IPython.display import Image

Image(url="system_diagram.png", height=250)

Full absolute state vector:

$$
x = \begin{bmatrix}
x_f \\ v_f \\ x_e \\ v_e \\ x_b \\ v_b
\end{bmatrix}
$$

The controller observes only relative distances and velocity differences:

$$
d_{\text{front}} = x_f - x_e,\qquad
d_{\text{back}}  = x_e - x_b
$$

$$
v_{\text{front}} = v_f - v_e,\qquad
v_{\text{back}}  = v_e - v_b
$$

These are assembled as:

$$
u_{\text{NN}} = C_{\text{map}} \, x
$$

Where:

$$ C_{map} = 
\begin{bmatrix}
1 & 0 & -1 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & -1 & 0 \\
0 & 1 & 0 & -1 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 & -1
\end{bmatrix}
$$

Our 6D interval set $R_0$, which will be propagated through the system dynamics:
```matlab
R0 = interval( ...
    [100; 5; 30; 5; 0; 5], ...
    [100+1; 5+0.1; 30+1; 5+0.1; 0+1; 5+0.1] ...
    );
```

## 2. Normalisation

$$
\tilde{u} = \frac{u - \mu}{\sigma}
$$

Mapping from state $x$ to normalised NN inputs:

$$
\tilde{u} = M_{\text{nn}} x + k_{\text{nn}}
$$

where:

$$
M_{\text{nn}} = \mathrm{diag}(1/\sigma) \, C_{\text{map}},
\qquad
k_{\text{nn}} = -\mu/\sigma
$$


## 3. Continuous-time dynamics

We model a simple scenario:
- $a_f = -2$
- $a_b = 0$
- $a_e$ determined by the NN (discrete: brake/idle/accel)

The system is defined by:
$$\dot{x} = Ax + Bu + C$$

The following holds:

$$
\dot{x}_f = v_f = x_2, \qquad \dot{v}_f = a_f = -2
$$
$$
\dot{x}_e = v_e = x_4, \qquad \dot{v}_e = a_e = u
$$
$$
\dot{x}_b = v_b = x_6, \qquad \dot{v}_b = a_b = 0
$$

Thus:

$$ A = 
\begin{bmatrix}
0 & 1 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 1 \\
0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix},

B = \begin{bmatrix}
0 \\ 0 \\ 0 \\ 1 \\ 0 \\ 0
\end{bmatrix}, 

C = \begin{bmatrix}
0 \\ -2 \\ 0 \\ 0 \\ 0 \\ 0
\end{bmatrix}
$$

System 1 brake ($a_{ego}=-5$):
$$ c_{brake} = c + B \cdot actions(1) $$

System 2 idle ($a_{ego}=-0$):
$$ c_{idle} = c + B \cdot actions(2) $$

System 3 accelerate ($a_{ego}=3$):
$$ c_{acc} = c + B \cdot actions(3) $$

## 4. Unsafe sets

Front safety:

$$
x_f - x_e - T_{\text{gap}} v_e \ge D_{\text{default}}
$$

Back safety:

$$
x_e - x_b \ge D_{\text{default}}
$$

We convert these into halfspaces:

$$
H x \le k
$$

Where:
$$
H_{unsafe\_front} = \begin{bmatrix}
1 \\ 0 \\ -1 \\ -T_{gap} \\ 0 \\ 0
\end{bmatrix}, 

H_{unsafe\_back} = \begin{bmatrix}
0 \\ 0 \\ 1 \\ 0 \\ -1 \\ 0
\end{bmatrix}, 
$$

## 5. Reachability loop
We iterate:
1. Map reachable set to NN input
2. Compute NN output region
3. Select discrete action using interval argmax
4. Propagate reachability through continuous dynamics
5. Check unsafe halfspace intersections